In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!nvidia-smi

Mon Jun 15 18:31:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install transformers

In [5]:
from transformers import AutoTokenizer

# Load the tokenizer for a specific model (e.g., GPT-2)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Tokenize some input text
text = "Hello, how are you?"
tokens = tokenizer(text, return_tensors='pt')
print(tokens)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'input_ids': tensor([[15496,    11,   703,   389,   345,    30]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [6]:
from transformers import AutoModelForCausalLM

# Load the pre-trained GPT-2 model
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Generate text
input_ids = tokenizer.encode("indian cricket", return_tensors='pt')
output = model.generate(input_ids, max_length=50)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print(generated_text)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


indian cricket team, which has been in the country for over a decade.

The team's captain, Ravi Shankar, has been in the country for over a decade.

The team's captain, Ravi Shankar,


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load a pre-trained model and tokenizer
model_name = "gpt2"  # You can replace with any other LLM
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def chunk_text(text, max_length=512):
    """Chunk text into smaller pieces."""
    tokens = tokenizer.encode(text, return_tensors='pt')[0]
    chunks = []

    for i in range(0, len(tokens), max_length):
        chunk = tokens[i:i + max_length]
        chunks.append(chunk)

    return chunks

def generate_responses(chunks):
    """Generate responses for each chunk using the LLM."""
    responses = []
    for chunk in chunks:
        input_ids = chunk.unsqueeze(0)  # Add batch dimension
        # Increase max_length to a value greater than or equal to the longest chunk length
        output = model.generate(input_ids, max_length=512)  # Generate response
        responses.append(tokenizer.decode(output[0], skip_special_tokens=True))

    return responses

# Example long text
long_text = "india " * 5  # Repeat to simulate long text

# Chunk the text
chunks = chunk_text(long_text)

# Generate responses for each chunk
responses = generate_responses(chunks)

# Print the responses
for i, response in enumerate(responses):
    print(f"Response for chunk {i+1}:\n{response}\n")

print(responses)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Response for chunk 1:
india india india india india ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia

['india india india india india ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia ia 

In [9]:
# INtroduce to Hugging face
# import the model
# refer for hugging face --> https://huggingface.co/openai-community/gpt2

from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')
set_seed(42)
generator("Hello, I'm a language model,", max_length=30, num_return_sequences=5)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_

[{'generated_text': "Hello, I'm a language model, so I can write things that are easy to understand with a little bit of code. But when you think of words, it's hard to think of words that are as simple as a little word in a sentence.\n\nSo I'm going to use a little bit of code, and I'll talk a little bit about the syntax.\n\nThis is how you write your first line:\n\n$trans = new $trans [1, 2, 3, 4]; $trans -> write ( 'Hello, I'm a language model, so I can write things that are easier to understand with a little bit of code.' );\n\nThis code is pretty simple, but it really doesn't have to be. We want to write this code in a few lines, and we're going to use an expression, which is a shorthand for the literal of the language.\n\nIn this case, we're going to use a variable named trans that's a symbol. We want to write this expression as an expression, where we're going to look for a line that matches the line we want to write.\n\nThe syntax for writing a line like this is very simple:\n\

In [10]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')
set_seed(42)
generator("Hello, explain about indian economy ,", max_length=10, num_return_sequences=2)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=10) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Hello, explain about indian economy , for you understand that the Indian economy is highly dependent on the Indian Government . The Indian Government provides for its citizens with a comprehensive income tax, and this income tax is not charged in India , but it is charged on a monthly basis by the Indians. The Indian Government also charges tax on the income of foreign citizens. In other words, Indian citizens can do their own tax, but they could not do the same as Indian citizens in the United States where they are subject to the taxes imposed by the Indian Government . This is also why Indians are not allowed to move around freely .\n\nIn the United States , your Indian citizenship is not subject to taxation, but you own the land and the people of the United States .\n\nThe Indians do not have to pay any income tax on their own , they can move freely throughout the country .\n\n, they can move freely throughout the country . In other words, the Indian citizens ca

In [11]:
from transformers import GPT2Tokenizer, GPT2Model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2Model.from_pretrained('gpt2')
text = "Replace me by any text you'd like."
encoded_input = tokenizer(text, return_tensors='pt')
output = model(**encoded_input)
print(output)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=tensor([[[ 0.1629, -0.2166, -0.1410,  ..., -0.2619, -0.0819,  0.0092],
         [ 0.4628,  0.0248, -0.0785,  ..., -0.0859,  0.5122, -0.3939],
         [-0.0644,  0.1551, -0.6306,  ...,  0.2488,  0.3691,  0.0833],
         ...,
         [-0.5591, -0.4490, -1.4540,  ...,  0.1650, -0.1302, -0.3740],
         [ 0.1400, -0.3875, -0.7916,  ..., -0.1780,  0.1824,  0.2185],
         [ 0.1721, -0.2420, -0.1124,  ..., -0.1068,  0.1205, -0.3213]]],
       grad_fn=<ViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)


In [15]:
!pip install transformers tensorflow -q

In [18]:
!pip install tensorflow
!pip install torch
!pip install sentencepiece

In [30]:
from transformers import GPT2Tokenizer, GPT2Model

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2Model.from_pretrained("gpt2")

text = "Replace me by any text you'd like."

inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

print(outputs)



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=tensor([[[ 0.1629, -0.2166, -0.1410,  ..., -0.2619, -0.0819,  0.0092],
         [ 0.4628,  0.0248, -0.0785,  ..., -0.0859,  0.5122, -0.3939],
         [-0.0644,  0.1551, -0.6306,  ...,  0.2488,  0.3691,  0.0833],
         ...,
         [-0.5591, -0.4490, -1.4540,  ...,  0.1650, -0.1302, -0.3740],
         [ 0.1400, -0.3875, -0.7916,  ..., -0.1780,  0.1824,  0.2185],
         [ 0.1721, -0.2420, -0.1124,  ..., -0.1068,  0.1205, -0.3213]]],
       grad_fn=<ViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)


In [14]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')

set_seed(42)
generator("The White man worked as a", max_length=10, num_return_sequences=5)


set_seed(42)
generator("The Black man worked as a", max_length=10, num_return_sequences=5)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=10) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=10) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'The Black man worked as a clerk at the warehouse and was paid $1,500 a month to do the work.\n\nWhen the black man\'s employer, the company that owned the warehouse, began selling slaves, he received a paycheck that was close to $1,000.\n\nThe company did not dispute the report, and the company did not tell the city that it was not paying back the black man.\n\nBut the city\'s police chief, Bill Williams, said he did not hear anything from the black man, and the city did not know what was going on.\n\n"I think this is the kind of thing that would be a hard decision for the city to make, for us to look at," Williams said.\n\nWilliams said he expects that from the city to respond to the report.\n\n"I think (police) will go to the city and look at it and see what they\'ve got," Williams said.\n\nA city spokesman did not immediately respond to a request for comment.\n\nAccording to a Department of Justice report last year, the state of Mississippi has made it illegal t